# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Muhammad Islam

**ID**: mai58

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Desktop/hw5-mai58hw5`
   Installed Pango_jll ────── v1.57.0+0
   Installed PlotUtils ────── v1.4.4
   Installed Measures ─────── v0.3.3
   Installed GraphRecipes ─── v0.5.15
   Installed StatsBase ────── v0.34.8
   Installed DataStructures ─ v0.19.3
   Installed JSON ─────────── v1.3.0
   Installed Adapt ────────── v4.4.0
   Installed StableRNGs ───── v1.0.4
   Installed StructUtils ──── v2.6.0
   Installed JuMP ─────────── v1.29.3
   Installed HiGHS ────────── v1.20.1
   Installed ForwardDiff ──── v1.3.0
Precompiling project...
    457.4 ms  ✓ Measures
    505.2 ms  ✓ StableRNGs
    588.7 ms  ✓ Adapt
    761.0 ms  ✓ StructUtils
   1109.1 ms  ✓ ChainRulesCore
    767.0 ms  ✓ ColorVectorSpace → SpecialFunctionsExt
   1554.8 ms  ✓ DataStructures
    497.8 ms  ✓ OffsetArrays → OffsetArraysAdaptExt
    868.5 ms  ✓ Adapt → AdaptSparseArraysExt
   1410.2 ms  ✓ Pango_jll
    657.6 ms  ✓ StructUtils → StructUtilsTablesExt
    741.7 ms  ✓ ChainRulesCore → ChainRulesCo

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [3]:
# Waste composition (%), recycling rates (%), ash fractions (%)
w = [15, 40, 5, 3, 2, 5, 18, 4, 2, 2, 1, 3]
r = [0, 55, 15, 10, 0, 30, 40, 60, 75, 80, 50, 0]
a = [8, 7, 5, 10, 15, 2, 2, 100, 100, 100, 100, 70]

# Compute fractions
recycling_fraction = sum(w .* r) / 10000
ash_fraction        = sum(w .* a) / 10000

(round(recycling_fraction, digits=4),
 round(ash_fraction, digits=4))


(0.3775, 0.1641)

### Problem 1.1 – Overall Recycling and Ash Fractions

Based on the waste composition data, the overall recycling and ash fractions are computed by multiplying each component’s mass percentage by its corresponding recycling rate (for the MRF) or combustion ash content (for the WTE). Summing these contributions gives the total fraction of material that can be recycled and the fraction that becomes ash after combustion. Since all three cities have identical waste composition, these fractions apply equally to each city and will be used later when building the mass-balance constraints in the optimization model.

**Overall Recycling Fraction**

$$
f_R = \sum_i \left(\frac{w_i}{100}\right)\left(\frac{r_i}{100}\right) = 0.3775
$$

**Overall Ash Fraction**

$$
f_A = \sum_i \left(\frac{w_i}{100}\right)\left(\frac{a_i}{100}\right) = 0.1641
$$

**Final Answers (rounded):**
- Recycling fraction: **0.378**
- Ash fraction: **0.164**


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

### Problem 1.2 – Decision Variables

To formulate the mixed-integer optimization model, we define two types of decision variables: (1) waste-flow allocation variables and (2) binary facility-activation variables.

#### Waste-Flow Variables

- $x_{i,f}$: Continuous variable representing the amount of waste (Mg/day) transported from **city $i$** to **facility $f$**,  
  where $i \in \{1,2,3\}$ and $f \in \{\text{LF},\ \text{MRF},\ \text{WTE}\}$.

These variables determine how the waste generated in each city is allocated across different disposal options.

#### Facility-Activation Variables

- $y_f$: Binary variable indicating whether facility $f$ is opened and operated:

$$
y_f =
\begin{cases}
1, & \text{if facility } f \text{ operates} \\
0, & \text{otherwise}
\end{cases}
$$

where $f \in \{\text{LF},\ \text{MRF},\ \text{WTE}\}$.

These variables ensure that each facility incurs its fixed cost only if it is activated and enforce capacity limits based on whether the facility is open.

#### Summary of Decision Variables

- $x_{i,f} \ge 0$: Waste shipped from city $i$ to facility $f$ (Mg/day).  
- $y_f \in \{0,1\}$: Indicates whether facility $f$ is active.

Together, these variables fully describe the system’s waste flows and facility operation decisions.


#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

### Problem 1.3 – Objective Function

The goal of the optimization is to **minimize the total daily system cost** of handling municipal solid waste. This total cost includes:

1. Transportation cost from each city to each facility  
2. Tipping (processing) cost at each facility  
3. Recycling-processing cost at the MRF  
4. Fixed operating cost of each facility (if it is opened)

We use the following notation:

- $x_{i,f}$: waste (Mg/day) shipped from city $i$ to facility $f$  
- $y_f \in \{0,1\}$: $1$ if facility $f$ is opened  
- $i \in \{1,2,3\}$, $f \in \{\text{LF}, \text{MRF}, \text{WTE}\}$  
- $d_{i,f}$: distance (km) from city $i$ to facility $f$  
- $c^{\text{trans}} = 1.5\ \$/(\text{Mg}\cdot\text{km})$  
- $c^{\text{tip}}_{\text{LF}} = 50$, $c^{\text{tip}}_{\text{MRF}} = 7$, $c^{\text{tip}}_{\text{WTE}} = 60$  
- $c^{\text{rec}} = 40\ \$/\text{Mg}$  
- $F_{\text{LF}} = 2000$, $F_{\text{MRF}} = 1500$, $F_{\text{WTE}} = 2500$  
- $f_R = 0.3775$: overall recycling fraction (from Problem 1.1)

---

### Derivation

#### **1. Transportation + tipping cost**

Waste shipped from city $i$ to facility $f$ incurs transportation and tipping charges.  
For a flow $x_{i,f}$, the cost is:

$$
\left(c^{\text{trans}} d_{i,f} + c^{\text{tip}}_f\right)x_{i,f}.
$$

The total transportation + tipping cost is:

$$
\sum_{i=1}^3 \sum_{f \in \{\text{LF},\text{MRF},\text{WTE}\}}
\left(c^{\text{trans}} d_{i,f} + c^{\text{tip}}_f\right)x_{i,f}.
$$

---

#### **2. Recycling-processing cost at the MRF**

Only waste sent to the MRF has recyclable components.  
Total recycled mass is:

$$
f_R \sum_{i=1}^3 x_{i,\text{MRF}}.
$$

Cost of recycling:

$$
c^{\text{rec}} f_R \sum_{i=1}^3 x_{i,\text{MRF}}.
$$

---

#### **3. Fixed facility costs**

A facility incurs a fixed cost only if it is opened.  
Total fixed cost:

$$
\sum_{f \in \{\text{LF},\text{MRF},\text{WTE}\}} F_f y_f.
$$

---

### **Full Objective Function**

Combining all cost components, the objective is:

$$
\min Z =
\sum_{i=1}^3 \sum_{f \in \{\text{LF},\text{MRF},\text{WTE}\}}
\left(c^{\text{trans}} d_{i,f} + c^{\text{tip}}_f\right)x_{i,f}
\;+\;
c^{\text{rec}} f_R \sum_{i=1}^3 x_{i,\text{MRF}}
\;+\;
\sum_{f \in \{\text{LF},\text{MRF},\text{WTE}\}} F_f y_f.
$$

This objective function minimizes the total daily cost of transporting, processing, and disposing of municipal solid waste while accounting for recycling and facility opening costs.


#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

### Problem 1.4 – Constraints

To complete the mixed-integer model, we derive constraints for mass balances, facility capacities, and variable domains. We also need to account for recycling at the MRF and ash production at the WTE, using the overall recycling and ash fractions from Problem 1.1.

We keep the decision variables from Problems 1.2–1.3:

- $x_{i,f}$: waste (Mg/day) shipped from city $i$ to facility $f$,  
  $i \in \{1,2,3\}$, $f \in \{\text{LF}, \text{MRF}, \text{WTE}\}$  
- $y_f \in \{0,1\}$: indicates whether facility $f$ is open

In addition, to track what happens to the residual waste from the MRF we introduce:

- $r_{\text{LF}}$: MRF residuals sent to the landfill (Mg/day)  
- $r_{\text{WTE}}$: MRF residuals sent to the WTE (Mg/day)

Let:

- $G_1 = 100$, $G_2 = 90$, $G_3 = 120$ (Mg/day): waste generated by each city  
- $\text{Cap}_{\text{LF}} = 200$, $\text{Cap}_{\text{MRF}} = 350$, $\text{Cap}_{\text{WTE}} = 210$ (Mg/day): facility capacities  
- $f_R = 0.3775$: overall recycling fraction for waste sent to the MRF (from Problem 1.1)  
- $f_A^{\text{orig}} = 0.1641$: overall ash fraction for original MSW (from Problem 1.1)  
- $f_A^{\text{res}}$: ash fraction of the **MRF residual** stream when it is sent to the WTE (computed from the composition of the non-recycled material)

---

#### 1. City mass-balance constraints

All waste generated in each city must be sent to one of the three facilities:

$$
\sum_{f \in \{\text{LF}, \text{MRF}, \text{WTE}\}} x_{i,f} = G_i,
\qquad i = 1,2,3.
$$

These constraints ensure that there is no accumulation or loss of waste at the city level.

---

#### 2. MRF mass balance and residual split

Let the total mass entering the MRF be $\sum_{i=1}^3 x_{i,\text{MRF}}$.

- Recycled mass: $f_R \sum_{i=1}^3 x_{i,\text{MRF}}$  
- Residual mass leaving the MRF: $(1 - f_R) \sum_{i=1}^3 x_{i,\text{MRF}}$

We allow the residuals to be split between the landfill and the WTE:

$$
r_{\text{LF}} + r_{\text{WTE}} = (1 - f_R)\sum_{i=1}^3 x_{i,\text{MRF}}.
$$

This constraint conserves mass around the MRF.

---

#### 3. Facility capacity constraints

Each facility can process only up to its capacity **if it is open** (i.e., $y_f = 1$).

- **MRF capacity (based on inflow of original waste):**

$$
\sum_{i=1}^3 x_{i,\text{MRF}} \le \text{Cap}_{\text{MRF}}\, y_{\text{MRF}}
= 350\, y_{\text{MRF}}.
$$

- **WTE capacity (total feed = direct waste + MRF residuals to WTE):**

$$
\sum_{i=1}^3 x_{i,\text{WTE}} + r_{\text{WTE}}
\le \text{Cap}_{\text{WTE}}\, y_{\text{WTE}}
= 210\, y_{\text{WTE}}.
$$

- **Landfill capacity (total disposed mass):**

The landfill receives:

1. Direct waste from cities: $\sum_{i=1}^3 x_{i,\text{LF}}$  
2. MRF residuals sent directly to landfill: $r_{\text{LF}}$  
3. Combustion ash from the WTE:
   - Ash from direct WTE feed: $f_A^{\text{orig}}\sum_{i=1}^3 x_{i,\text{WTE}}$  
   - Ash from residuals burned at the WTE: $f_A^{\text{res}}\, r_{\text{WTE}}$

Thus, the landfill capacity constraint is:

$$
\sum_{i=1}^3 x_{i,\text{LF}}
+ r_{\text{LF}}
+ f_A^{\text{orig}}\sum_{i=1}^3 x_{i,\text{WTE}}
+ f_A^{\text{res}}\, r_{\text{WTE}}
\le \text{Cap}_{\text{LF}}\, y_{\text{LF}}
= 200\, y_{\text{LF}}.
$$

This ensures the landfill operates within its daily capacity and properly accounts for the ash that must be disposed there.

---

#### 4. Variable domain constraints

All flow and residual variables must be nonnegative:

$$
x_{i,f} \ge 0
\quad \forall i \in \{1,2,3\},\ f \in \{\text{LF}, \text{MRF}, \text{WTE}\},
$$

$$
r_{\text{LF}} \ge 0, \qquad r_{\text{WTE}} \ge 0.
$$

Facility activation variables are binary:

$$
y_f \in \{0,1\},
\qquad f \in \{\text{LF}, \text{MRF}, \text{WTE}\}.
$$

---

Together, these constraints enforce:

- Conservation of waste mass at the city, MRF, WTE, and landfill levels  
- Proper accounting for recycling and ash generation  
- Capacity limitations tied to facility activation  
- Physically meaningful (nonnegative) flow variables and binary open/closed decisions

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [15]:
using JuMP
using HiGHS

# ===========================
# Sets and indices
# ===========================
cities     = 1:3                 # 1,2,3
facilities = 1:3                 # 1:LF, 2:MRF, 3:WTE

LF  = 1       # landfill index
MRF = 2       # MRF index
WTE = 3       # WTE index

# ===========================
# Data
# ===========================

# Waste generated (Mg/day)
G = [100.0, 90.0, 120.0]         # city 1,2,3

# Capacities (Mg/day)
cap = Dict(
    LF  => 200.0,
    MRF => 350.0,
    WTE => 210.0,
)

# Distances from cities to facilities (km)
d_city = [
    5.0   30.0  15.0;
    15.0  25.0  10.0;
    13.0  45.0  20.0
]

# Distances between facilities (km)
d_fac = [
     0.0  32.0 18.0;
    32.0   0.0 15.0;
    18.0  15.0  0.0
]

c_trans = 1.5
c_tip   = Dict(LF => 50.0, MRF => 7.0, WTE => 60.0)
c_rec   = 40.0
F       = Dict(LF => 2000.0, MRF => 1500.0, WTE => 2500.0)

# Fractions from 1.1
f_R = 0.3775
f_A = 0.1641

# ===========================
# Model
# ===========================
model = Model(HiGHS.Optimizer)

@variable(model, x[cities, facilities] >= 0)
@variable(model, y_MRF_LF  >= 0)
@variable(model, y_MRF_WTE >= 0)
@variable(model, y_WTE_LF  >= 0)
@variable(model, delta[facilities], Bin)

# ===========================
# Constraints
# ===========================

@constraint(model, [i in cities],
    sum(x[i,j] for j in facilities) == G[i]
)

@constraint(model,
    y_MRF_LF + y_MRF_WTE == (1 - f_R)*sum(x[i,MRF] for i in cities)
)

@constraint(model,
    y_WTE_LF == f_A * (sum(x[i,WTE] for i in cities) + y_MRF_WTE)
)

@constraint(model,
    sum(x[i,MRF] for i in cities) <= cap[MRF]*delta[MRF]
)

@constraint(model,
    sum(x[i,WTE] for i in cities) + y_MRF_WTE <= cap[WTE]*delta[WTE]
)

@constraint(model,
    sum(x[i,LF] for i in cities) + y_MRF_LF + y_WTE_LF <= cap[LF]*delta[LF]
)

# ===========================
# Objective
# ===========================

@expression(model,
    cost_city_fac,
    sum((c_trans * d_city[i,j] + c_tip[j]) * x[i,j]
    for i in cities, j in facilities)
)

@expression(model,
    cost_recycle,
    c_rec * f_R * sum(x[i,MRF] for i in cities)
)

@expression(model,
    cost_residual,
    (c_trans * d_fac[MRF,LF]  + c_tip[LF])  * y_MRF_LF +
    (c_trans * d_fac[MRF,WTE] + c_tip[WTE]) * y_MRF_WTE
)

@expression(model,
    cost_ash,
    (c_trans * d_fac[WTE,LF] + c_tip[LF]) * y_WTE_LF
)

@expression(model,
    cost_fixed,
    sum(F[j] * delta[j] for j in facilities)
)

@objective(model, Min, cost_city_fac + cost_recycle + cost_residual + cost_ash + cost_fixed)

set_silent(model)
optimize!(model)

# ===========================
# Output
# ===========================

println("Status: ", termination_status(model))
println("Objective value: \$", round(objective_value(model), digits=2), "\n")

println("Flows x[i,j]:")
for i in cities, j in facilities
    if value(x[i,j]) > 1e-6
        println("  x[$i,$j] = ", round(value(x[i,j]), digits=3))
    end
end
println()

println("Residual & ash flows:")
println("  y_MRF_LF  = ", round(value(y_MRF_LF), digits=3))
println("  y_MRF_WTE = ", round(value(y_MRF_WTE), digits=3))
println("  y_WTE_LF  = ", round(value(y_WTE_LF), digits=3), "\n")

println("Facility activation:")
println("  delta_LF  = ", Int(round(value(delta[LF]))))
println("  delta_MRF = ", Int(round(value(delta[MRF]))))
println("  delta_WTE = ", Int(round(value(delta[WTE]))))


Status: OPTIMAL
Objective value: $27855.48

Flows x[i,j]:
  x[1,1] = 100.0
  x[2,3] = 90.0
  x[3,1] = 78.405
  x[3,3] = 41.595

Residual & ash flows:
  y_MRF_LF  = 0.0
  y_MRF_WTE = 0.0
  y_WTE_LF  = 21.595

Facility activation:
  delta_LF  = 1
  delta_MRF = 0
  delta_WTE = 1


### Problem 1.5 — Optimal Solution

Using the mixed–integer linear programming model developed in Problems 1.2–1.4, we solved the waste management system using **JuMP** and the **HiGHS** MILP solver. The model incorporates transportation costs, tipping fees, recycling processing costs, ash handling costs, and fixed facility activation costs.

The solver returns the following **minimum total daily system cost**:

$$
\boxed{27,855.48\ \text{\$}/\text{day}}
$$

---

### Optimal Waste Flows (Mg/day)

The optimal city–to–facility flows $x_{i,f}$ are:

| City → Facility | LF | MRF | WTE |
|-----------------|----|-----|-----|
| **City 1** | $100.000$ | $0$ | $0$ |
| **City 2** | $0$ | $0$ | $90.000$ |
| **City 3** | $78.405$ | $0$ | $41.595$ |

Interpretation:

- City 1 sends all waste to **LF**.  
- City 2 sends all waste to **WTE**.  
- City 3 splits its waste between LF and WTE.  
- The **MRF is unused** because it never becomes cost-optimal.

---

### Residual and Ash Flows

Let:

- $y_{\text{MRF}\to\text{LF}}$ = MRF residuals sent to LF  
- $y_{\text{MRF}\to\text{WTE}}$ = MRF residuals sent to WTE  
- $y_{\text{WTE}\to\text{LF}}$ = ash from WTE sent to LF  

The model returns:

$$
\begin{aligned}
y_{\text{MRF}\to\text{LF}} &= 0, \\
y_{\text{MRF}\to\text{WTE}} &= 0, \\
y_{\text{WTE}\to\text{LF}} &= 21.595.
\end{aligned}
$$

Since no waste is processed at the MRF, no residuals are produced.  
The WTE receives $131.595\ \text{Mg/day}$ and generates $21.595\ \text{Mg/day}$ of ash.

---

### Facility Activation Decisions

The binary activation variables $\delta_f$ yield:

| Facility | $\delta_f$ |
|----------|------------|
| LF | $1$ |
| MRF | $0$ |
| WTE | $1$ |

- **LF** is open and receives waste plus all WTE ash.  
- **WTE** is open and processes waste from Cities 2 and 3.  
- **MRF** remains closed because using it does not reduce system cost.

---

### Summary

The optimal solution is that the WTE facility processes all $90\ \text{Mg/day}$ of waste from City 2, plus an additional $41.595\ \text{Mg/day}$ from City 3. This gives WTE a total feed of $131.595\ \text{Mg/day}$ and produces approximately $21.595\ \text{Mg/day}$ of ash, all of which is transported to the landfill. In addition to receiving the ash, the landfill also accepts $100\ \text{Mg/day}$ of waste from City 1 and $78.405\ \text{Mg/day}$ from City 3, reaching exactly its $200\ \text{Mg/day}$ capacity. The MRF is completely unused in the optimal configuration. The resulting minimum total daily system cost is: $$ Z^\ = 27,855.48\ \text{\$}/\text{day}.$$


#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

In [22]:
println("""
                   ┌────────┐
                   │ City 1 │
                   └────┬───┘
                        │ 100 Mg/day
                        ▼
                   ┌──────────┐
                   │LF 200 Mg │◄───────────────┐
                   └────┬─────┘                │
                        ▲                      │
          78.405 Mg/day │                      │ 21.595 Mg/day ash
   ┌────────┐           │                      │
   │ 120    │           │                      │
   │ Mg/day │           │                      │
   │ City 3 │───────────┘                      │
   └───┬────┘                                  │
       │ 41.595 Mg/day                         │
       ▼                                       │
   ┌───────────┐     90 Mg/day                 │
   │WTE 131.595│◄──────┐                       │
   └───────────┘       │                       │
                       │                       │
                   ┌────────┐                  │
                   │ City 2 │──────────────────┘
                   └────────┘

MRF is unused (no flows).
""")


                   ┌────────┐
                   │ City 1 │
                   └────┬───┘
                        │ 100 Mg/day
                        ▼
                   ┌──────────┐
                   │LF 200 Mg │◄───────────────┐
                   └────┬─────┘                │
                        ▲                      │
          78.405 Mg/day │                      │ 21.595 Mg/day ash
   ┌────────┐           │                      │
   │ 120    │           │                      │
   │ Mg/day │           │                      │
   │ City 3 │───────────┘                      │
   └───┬────┘                                  │
       │ 41.595 Mg/day                         │
       ▼                                       │
   ┌───────────┐     90 Mg/day                 │
   │WTE 131.595│◄──────┐                       │
   └───────────┘       │                       │
                       │                       │
                   ┌────────┐                  │
             

### Problem 1.6 — Flow Diagram and Interpretation

The flow diagram shows the optimal movement of waste from the cities to the facilities. City 1 sends all of its waste ($100\ \text{Mg/day}$) to the landfill (LF), while City 2 sends all of its waste ($90\ \text{Mg/day}$) to the waste-to-energy facility (WTE). City 3 splits its waste, sending $78.405\ \text{Mg/day}$ to LF and $41.595\ \text{Mg/day}$ to WTE. The WTE processes a total of $131.595\ \text{Mg/day}$ and generates $21.595\ \text{Mg/day}$ of ash, which is transported to LF. This brings the landfill’s total intake to $200\ \text{Mg/day}$, exactly matching its capacity. The **MRF is not used at all** in the optimal solution,no waste is sent to it, and it processes no material.

This solution makes sense because sending waste to the MRF would introduce additional costs from recycling processing and handling of MRF residuals, without reducing total system cost. Since the LF and WTE both have sufficient capacity to receive all waste and ash, and their combined operating costs are lower than involving the MRF, the model correctly identifies that the MRF should remain unused to achieve the minimum total cost.


### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is $1500 \text{MW}$
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

### Problem 2.1 — Scenario Tree

In this two-period economic dispatch problem, period 1 is deterministic:

- Period 1 demand: $d_1 = 1100\ \text{MW}$
- Period 1 solar CF: $0.9$
- Period 1 wind CF: $0.45$

Uncertainty appears only in **period 2**, in both demand and renewable capacity factors.

- Period 2 demand:
  - $d_2 = 1200\ \text{MW}$ with probability $0.75$
  - $d_2 = 1500\ \text{MW}$ with probability $0.25$
- Period 2 solar and wind CFs:
  - $(\text{solar}, \text{wind}) = (0.95, 0.40)$ with probability $0.70$
  - $(\text{solar}, \text{wind}) = (0.75, 0.50)$ with probability $0.30$

Assuming demand and renewable outcomes are independent, we obtain **four terminal scenarios** at period 2:

1. $d_2 = 1200,\ (\text{solar},\text{wind}) = (0.95, 0.40)$,  
   probability $0.75 \times 0.70 = 0.525$.
2. $d_2 = 1200,\ (\text{solar},\text{wind}) = (0.75, 0.50)$,  
   probability $0.75 \times 0.30 = 0.225$.
3. $d_2 = 1500,\ (\text{solar},\text{wind}) = (0.95, 0.40)$,  
   probability $0.25 \times 0.70 = 0.175$.
4. $d_2 = 1500,\ (\text{solar},\text{wind}) = (0.75, 0.50)$,  
   probability $0.25 \times 0.30 = 0.075$.

A simple ASCII scenario tree is:

```text
                        Period 1: t = 1
       Root node:  d1 = 1100, solar CF = 0.9, wind CF = 0.45
                                |
                                |
                    -------------------------
                     |                     |
      d2 = 1200 MW (p = 75%)       d2 = 1500 MW (p = 25%)
                     |                     |
      -------------------------         --------------------
        |                 |                 |             |
   Scenario 1         Scenario 2        Scenario 3     Scenario 4
   p = 52.5%          p = 22.5%         p = 17.5%      p = 7.5%
   d2 = 1200          d2 = 1200         d2 = 1500      d2 = 1500
   solar = 0.95       solar = 0.75      solar = 0.95   solar = 0.75
   wind  = 0.40       wind  = 0.50      wind  = 0.40   wind  = 0.50

The scenario tree for this problem captures how uncertainty unfolds between the two time periods. Period 1 is fully deterministic, with known demand and renewable capacity factors, so the branching occurs only in period 2. First, demand may be either 1200MW with probability 75% or 1500MW with probability 25%. Independently, the available renewable output in period 2 depends on whether solar and wind capacity factors follow a high-renewables case (0.95, 0.40) with probability 70% or a low-renewables case (0.75, 0.50) with probability 30%. Combining these uncertainties produces four terminal scenarios with probabilities equal to the product of their respective demand and renewable outcomes: 52.5%, 22.5%, 17.5%, and 7.5%. This structure reflects how dispatch decisions made in period 1 must anticipate several possible operating conditions in period 2, allowing the model to capture both load uncertainty and renewable variability.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

### Problem 2.2 — Stochastic Linear Program

We formulate a two-stage stochastic linear program using the scenario tree from Problem 2.1.  
Period 1 is deterministic, while period-2 decisions adapt to one of four possible scenarios.

Let:

- $\mathcal{G}$ = set of generators (from `data/generators.csv`)
- $\Omega = \{1,2,3,4\}$ = set of scenarios
- $p_\omega$ = probability of scenario $\omega$

From Problem 2.1:

$$
p_1 = 0.525,\quad
p_2 = 0.225,\quad
p_3 = 0.175,\quad
p_4 = 0.075.
$$

---

**Parameters**

Period-1 demand:

$$
d_1 = 1100\ \text{MW}.
$$

Period-2 scenario-dependent demand:

$$
d_2^\omega =
\begin{cases}
1200, & \omega = 1, 2,\\[4pt]
1500, & \omega = 3, 4.
\end{cases}
$$

Period-1 capacity factors:

$$
\alpha^{\text{solar}}_{1} = 0.9,
\qquad
\alpha^{\text{wind}}_{1} = 0.45.
$$

Period-2 capacity factors:

$$
(\alpha^{\text{solar}}_{2,\omega},\ \alpha^{\text{wind}}_{2,\omega}) =
\begin{cases}
(0.95,\, 0.40), & \omega = 1, 3,\\[4pt]
(0.75,\, 0.50), & \omega = 2, 4.
\end{cases}
$$

From `generators.csv`, each generator $g \in \mathcal{G}$ has:

- $c_g$: marginal cost  
- $\overline{P}_g$: capacity  
- $\underline{P}_g$: minimum output  
- $RU_g$: ramp-up limit  
- $RD_g$: ramp-down limit  

Define effective capacity factor:

$$
\gamma_{g,t,\omega} =
\begin{cases}
\alpha^{\text{solar}}_{t,\omega}, & g \text{ is solar},\\
\alpha^{\text{wind}}_{t,\omega}, & g \text{ is wind},\\
1, & g \text{ is conventional}.
\end{cases}
$$

---

**Decision Variables**

First-stage (period-1) generation:

$$
P_{g,1} \ge 0.
$$

Second-stage (period-2) generation:

$$
P_{g,2}^\omega \ge 0,
\qquad \forall \omega \in \Omega.
$$

---

**Objective Function**

Minimize total expected cost:

$$
\min Z
=
\sum_{g \in \mathcal{G}} c_g P_{g,1}
\;+\;
\sum_{\omega \in \Omega} p_\omega 
\sum_{g \in \mathcal{G}} c_g P_{g,2}^\omega.
$$

---

**Constraints**

**1. Power Balance**

Period 1:

$$
\sum_{g \in \mathcal{G}} P_{g,1} = d_1.
$$

Period 2 (scenario-dependent):

$$
\sum_{g \in \mathcal{G}} P_{g,2}^\omega = d_2^\omega,
\qquad \forall \omega \in \Omega.
$$

---

**2. Generator Capacity Limits**

Period 1:

$$
\underline{P}_g
\;\le\;
P_{g,1}
\;\le\;
\gamma_{g,1,\omega}\,\overline{P}_g,
\qquad \forall g \in \mathcal{G}.
$$

Period 2:

$$
\underline{P}_g
\;\le\;
P_{g,2}^\omega
\;\le\;
\gamma_{g,2,\omega}\,\overline{P}_g,
\qquad \forall g \in \mathcal{G},\ \forall \omega \in \Omega.
$$

---

**3. Ramping Constraints**

Coupling period-1 output to every period-2 scenario:

$$
P_{g,2}^\omega - P_{g,1} \le RU_g,
\qquad \forall g,\ \forall \omega,
$$

$$
P_{g,1} - P_{g,2}^\omega \le RD_g,
\qquad \forall g,\ \forall \omega.
$$

---

**4. Nonnegativity**

$$
P_{g,1} \ge 0,\qquad 
P_{g,2}^\omega \ge 0,
\quad \forall g,\ \forall \omega.
$$

---

The two-stage stochastic linear program models generator dispatch under uncertainty in both load and renewable output in period 2. Since period 1 is deterministic, all generators choose their initial dispatch levels $P_{g,1}$ before any uncertainty is revealed. In period 2, generation levels $P_{g,2}^\omega$ adapt to one of four possible scenarios determined by the combinations of uncertain demand and solar/wind capacity factors. The objective minimizes total expected cost by summing period-1 costs and the probability-weighted costs of each scenario. Power balance constraints enforce supply–demand matching, while capacity constraints incorporate renewable variability through the scenario-dependent capacity factors $\gamma_{g,t,\omega}$. Ramping constraints link the two periods by limiting how much output can change between $P_{g,1}$ and $P_{g,2}^\omega$. This stochastic linear program ensures that period-1 decisions are made before uncertainty is realized, while period-2 generation adapts optimally across all four scenarios, providing a cost-minimizing and scenario-responsive dispatch strategy.



## References

Lecture Slides and ChatGPT.